# Compare SAL, RDD and STDWI in a scatter plot

In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
from tqdm import tqdm, trange

import symmnet
from symmnet.sal_net import SALNetBase, AlphaPSP
from symmnet.utils import asym_angle, corrcoef
from symmnet.rdd_net import RDDNetBase
from symmnet.stdwi_original import (
    correlated_poisson_spike_train,
    random_sample_spike_train,
    spike_trains_to_xpsps,
    lif_dynamics,
    stdwi_method,
    binary_spike_matrix,
    create_stdwi_trace,
)

In [ ]:
mpl.style.use("../mystyle.mpl")

In [ ]:
FIG_DIR = Path("figs/")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/stdwi")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
SIMULATE = False

In [ ]:
def plot_weights(fig, ax, w, b):
    boundary = max([np.max(np.abs(w)), np.max(np.abs(b))]) * 1.1
    ax.scatter(w, b, alpha=0.1)
    ax.set_xlim(-boundary, boundary)
    ax.set_ylim(-boundary, boundary)
    ax.grid()
    ax.set_xlabel("W")
    ax.set_ylabel("B")
    ax.set_aspect("equal")

In [ ]:
## general settings:
n_layers = 2
dims = [50, 50]  # [input, output] dimnension
n_in, n_out = dims[0], dims[1]

n_epochs = 100
dur_epoch = 1.0  # [s]

rng = np.random.default_rng(seed=43)

# draw weigts using kaiming he (uniform)
bound = 1.0 * np.sqrt(3.0 / n_in)
weight = rng.uniform(low=-bound, high=bound, size=dims[::-1])
fb_weight = rng.uniform(low=-bound, high=bound, size=dims)

# draw weigts using kaiming he (normal)
# std = 1.0 * np.sqrt(1.0 / n_in)
# weight = rng.normal(loc=0., scale=std**2, size=dims[::-1])
# fb_weight = rng.normal(loc=0., scale=std**2, size=dims)

In [ ]:
fig, ax = plt.subplots()
plot_weights(fig, ax, weight, fb_weight)

## SAL

In [ ]:
# SAL time units: milliseconds

params = {
    "t_ref": 10,  # [ms]
    "sal_lr": 0.05,
    "batchsize": 8,
    "t_syn": 10.0,
}

dt = 1.0 / params["t_ref"]  # [ms]
epoch_dur_ts = int(dur_epoch * 1e3 / (dt * params["batchsize"]))

device = "cuda" if torch.cuda.is_available() else "cpu"

salnet = SALNetBase(
    layer_dims=dims,
    t_ref=params["t_ref"],
    stdp_lr=params["sal_lr"],
    batch_size=params["batchsize"],
)
salnet.layers[1].weight.data = torch.from_numpy(weight.astype(np.float32))
salnet.layers[0].fb_weight.data = torch.from_numpy(fb_weight.astype(np.float32))
salnet.to(device)

if SIMULATE:

    sal_res = defaultdict(list)

    time = 0.0
    w = salnet.layers[1].weight.detach()
    b = salnet.layers[0].fb_weight.detach().t()
    sal_res["time"].append(time)
    sal_res["angle"].append(asym_angle(w, b).cpu().numpy())
    sal_res["corr"].append(corrcoef(w, b).cpu().numpy())

    with torch.no_grad():
        for i in trange(n_epochs, desc="SALNet epoch"):
            time += dur_epoch
            for t in range(epoch_dur_ts):
                salnet.update_mempot()
                salnet.update_spikes()
                salnet.fb_stdp_online()
            dw = salnet.apply_fb_weight_update()
            w = salnet.layers[1].weight.detach()
            b = salnet.layers[0].fb_weight.detach().t()
            sal_res["time"].append(time)
            sal_res["angle"].append(asym_angle(w, b).cpu().numpy())
            sal_res["corr"].append(corrcoef(w, b).cpu().numpy())

    fb_weight_sal = b.detach().cpu().numpy().flatten()

    with open(DATA_DIR / "sal.pkl", "wb") as f:

        sal_data = {
            "weight": weight,
            "fb_weight": fb_weight_sal,
            "res": sal_res,
        }
        pickle.dump(sal_data, f)

else:
    with open(DATA_DIR / "sal.pkl", "rb") as f:
        sal_data = pickle.load(f)
        weight = sal_data["weight"]
        fb_weight_sal = sal_data["fb_weight"]
        sal_res = sal_data["res"]

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(sal_res["time"], sal_res["angle"])
ax[1].plot(sal_res["time"], sal_res["corr"])

In [ ]:
fig, ax = plt.subplots()
plot_weights(fig, ax, weight, fb_weight_sal)

## RDD

In [ ]:
# setup the RDD network
# RDD time units: seconds
params = {
    "stim_ratio": 0.2,
    "rdd_time": dur_epoch,  # [s]
}

rdd_net = RDDNetBase(dims)
rdd_net.reset()
rdd_net.classification_layers[0].set_weights(fb_weight=fb_weight)
rdd_net.classification_layers[1].set_weights(weight=weight)

rng_rdd = np.random.default_rng(44)

In [ ]:
# compute the driving input:
indices_1 = rng_rdd.choice(a=n_in, size=int(params["stim_ratio"] * n_in), replace=False)
driving_rates_1 = np.zeros((n_in, 1))
driving_rates_1[indices_1] = symmnet.input_rate
driving_spike_hist_1 = np.zeros((n_in, symmnet.mem), dtype=int)
# spike_rates_1 = np.zeros(n_in)
# spike_rates_2 = np.zeros(n_out)

if SIMULATE:

    rdd_res = defaultdict(list)

    time = 0.0
    w = torch.from_numpy(weight)
    b = torch.from_numpy(rdd_net.classification_layers[0].fb_weight.T)
    rdd_res["time"].append(time)
    rdd_res["angle"].append(asym_angle(w, b))
    rdd_res["corr"].append(corrcoef(w, b))

    for epoch in trange(n_epochs):
        time += params["rdd_time"]
        for i in range(int(params["rdd_time"] / symmnet.dt)):
            if (i + 1) % (0.1 / symmnet.dt) == 0:
                indices_1 = rng_rdd.choice(
                    a=n_in, size=int(params["stim_ratio"] * n_in), replace=False
                )
                driving_rates_1 = np.zeros((n_in, 1))
                driving_rates_1[indices_1] = symmnet.input_rate
            driving_spike_hist_1 = np.concatenate(
                [driving_spike_hist_1[:, 1:], rng_rdd.poisson(driving_rates_1)], axis=1
            )

            rdd_net.out(driving_spike_hist_1)
            if (i + 1) % (10.0 / symmnet.dt) == 0:
                rdd_net.update_fb_weights()

                w = torch.from_numpy(weight)
                b = torch.from_numpy(rdd_net.classification_layers[0].fb_weight.T)
                rdd_res["time"].append(time)
                rdd_res["angle"].append(asym_angle(w, b).cpu().numpy())
                rdd_res["corr"].append(corrcoef(w, b).cpu().numpy())

    fb_weight_rdd = rdd_net.classification_layers[0].fb_weight.T

    with open(DATA_DIR / "rdd.pkl", "wb") as f:

        rdd_data = {
            "weight": weight,
            "fb_weight": fb_weight_rdd,
            "res": rdd_res,
        }
        pickle.dump(rdd_data, f)

else:
    with open(DATA_DIR / "rdd.pkl", "rb") as f:
        rdd_data = pickle.load(f)
        weight = rdd_data["weight"]
        fb_weight_rdd = rdd_data["fb_weight"]
        rdd_res = rdd_data["res"]

In [ ]:
fig, ax = plt.subplots()
plot_weights(fig, ax, weight, fb_weight_rdd)

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(rdd_res["time"], rdd_res["angle"])
ax[1].plot(rdd_res["time"], rdd_res["corr"])

## STDWI

In [ ]:
# STDWI time units: milliseconds

params = {
    "dt": 0.25,  # ms
    "stim_ratio": 0.2,
    "stim_correlation": 0.0,
    "stim_rate": 0.2,  # spks/ms
    "stim_duration": 100,  # ms
    "stim_weight_scale": 10,  # some random number I found in the STDWI code
    "weight_scale": 75,  # another random number I found in their code
    "tau_fast": 10.0,  # ms
    "tau_slow": 200.0,  # ms
    "lr": 2e-2,
    "decay_weighting": 0.1,  # again a random number that popped up in the code
}

DTYPE = np.float32

In [ ]:
if SIMULATE:
    sim_dur = n_epochs * dur_epoch * 1e3  # ms

    # stim --> input weights
    stim_input_weights = params["stim_weight_scale"] * np.eye(n_in)

    # input --> output weights
    weight_stdwi = params["weight_scale"] * np.sqrt(0.2 / params["stim_ratio"]) * weight
    weight_stdwi = np.astype(weight_stdwi, DTYPE)

    # stimulation spikes
    print("stimulation")
    stim_spks = correlated_poisson_spike_train(
        num_neurons=n_in,
        firing_rate=params["stim_rate"],
        correlation=params["stim_correlation"],
        simulation_time=sim_dur,
        timestep=params["dt"],
    )
    print("random sampling")
    stim_spks = random_sample_spike_train(
        stim_spks, sim_dur, params["dt"], params["stim_duration"], params["stim_ratio"]
    )

    # input neurons
    print("stim neurons xpsps")
    stim_xpsps = spike_trains_to_xpsps(stim_spks, sim_dur, params["dt"])
    print(stim_xpsps.dtype)
    print("input neuron spikes")
    inp_spks = lif_dynamics(stim_xpsps, stim_input_weights, params["dt"])
    del stim_xpsps

    # output neurons
    print("input neurons xpsps")
    inp_xpsps = spike_trains_to_xpsps(inp_spks, sim_dur, params["dt"])
    print("output neuron spikes")
    out_spks = lif_dynamics(inp_xpsps, weight_stdwi, params["dt"])
    print(out_spks[0].dtype)
    del inp_xpsps

    with open(DATA_DIR / "stdwi_spks.pkl", "wb") as f:
        pickle.dump((inp_spks, out_spks), f)

In [ ]:
# load spike data:
if False:
    with open(DATA_DIR / "stdwi_spks.pkl", "rb") as f:
        inp_spks, out_spks = pickle.load(f)

In [ ]:
# apply stdwi plasticity:

if SIMULATE:
    # My own leaner reimplementation of fitter.stdwi"""
    n_sim_timesteps = int(sim_dur / params["dt"])
    n_stims = int(sim_dur / params["stim_duration"])
    print(n_stims)

    a_fast = 1.0
    a_slow = a_fast * params["tau_fast"] / params["tau_slow"]

    # NOTE: from here on, the spikes will be converted into units of ms!
    if inp_spks[0].dtype == int:
        inp_spks = [spks * params["dt"] for spks in inp_spks]
        out_spks = [spks * params["dt"] for spks in out_spks]

    inp_spks_bin = binary_spike_matrix(inp_spks, sim_dur, params["dt"])
    out_spks_bin = binary_spike_matrix(out_spks, sim_dur, params["dt"])

    print(inp_spks_bin.shape)

    slow_input_trace = np.zeros((n_in, n_sim_timesteps), dtype=DTYPE)
    fast_input_trace = np.zeros_like(slow_input_trace)

    slow_input_trace = create_stdwi_trace(
        slow_input_trace,
        inp_spks_bin,
        a_slow,
        params["tau_slow"],
        params["dt"],
        alltoall=True,
    )
    fast_input_trace = create_stdwi_trace(
        fast_input_trace,
        inp_spks_bin,
        a_fast,
        params["tau_fast"],
        params["dt"],
        alltoall=True,
    )

    fb_weight_stdwi = np.astype(fb_weight.T, DTYPE)

    stdwi_res = defaultdict(list)

    time = 0.0
    w = torch.from_numpy(np.astype(weight, np.float32))
    b = torch.from_numpy(np.astype(fb_weight_stdwi, np.float32))
    stdwi_res["time"].append(time)
    stdwi_res["angle"].append(asym_angle(w, b))
    stdwi_res["corr"].append(corrcoef(w, b))

    # iterate over the stimulation periods
    len_stim = int(params["stim_duration"] / params["dt"])
    for i in trange(n_stims, desc="stdwi stimulation periods"):
        time += params["stim_duration"]
        # get trace chunks for the stim perdiod
        i_start = i * len_stim
        i_end = (i + 1) * len_stim
        chunk_slice = np.s_[
            :,
            i_start:i_end,
        ]
        # print(np.sum(out_spks_bin[chunk_slice]))
        fb_weight_stdwi = stdwi_method(
            fb_weight_stdwi,
            inp_spks_bin[chunk_slice],
            out_spks_bin[chunk_slice],
            slow_input_trace[chunk_slice],
            fast_input_trace[chunk_slice],
            params["lr"],
            decay_weighting=params["decay_weighting"],
        )
        # print(stdwi_fb_weight.dtype)

        w = torch.from_numpy(np.astype(weight, np.float32))
        b = torch.from_numpy(np.astype(fb_weight_stdwi, np.float32))
        stdwi_res["time"].append(time / 1e3)
        stdwi_res["angle"].append(asym_angle(w, b).cpu().numpy())
        stdwi_res["corr"].append(corrcoef(w, b).cpu().numpy())

    with open(DATA_DIR / "stdwi.pkl", "wb") as f:

        stdwi_data = {
            "weight": weight,
            "fb_weight": fb_weight_stdwi,
            "res": stdwi_res,
        }
        pickle.dump(stdwi_data, f)

else:
    with open(DATA_DIR / "stdwi.pkl", "rb") as f:
        stdwi_data = pickle.load(f)
        weight = stdwi_data["weight"]
        fb_weight_stdwi = stdwi_data["fb_weight"]
        stdwi_res = stdwi_data["res"]

In [ ]:
fig, ax = plt.subplots()
plot_weights(fig, ax, weight, fb_weight_stdwi)

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(stdwi_res["time"], stdwi_res["angle"])
ax[1].plot(stdwi_res["time"], stdwi_res["corr"])

## final plotting

In [ ]:
def calc_bound(w, b):
    return max([np.max(np.abs(w)), np.max(np.abs(b))]) * 1.1


def set_square_bound(ax, w, b):
    bound = calc_bound(w, b)
    ax.set_xlim(-bound, bound)
    ax.set_ylim(-bound, bound)


def sync_ticks(ax):
    ticks = ax.get_xticks()
    ylim = ax.get_ylim()
    ax.set_yticks(ticks[(ticks >= ylim[0]) & (ticks <= ylim[1])])
    ax.set_xticks(ticks[(ticks >= ylim[0]) & (ticks <= ylim[1])])


fig = plt.figure(figsize=(9.0 / 2.54, 8.0 / 2.54))

# --- Gap controls ---
gap_w = 0.35  # horizontal wspace
gap_h = 0.25  # vertical hspace
margin_left = 0.08
margin_right = 0.01
margin_top = 0.95
margin_bottom = 0.08

gs = gridspec.GridSpec(
    2,
    2,
    figure=fig,
    left=margin_left,
    right=1 - margin_right,
    top=margin_top,
    bottom=margin_bottom,
    wspace=gap_w,
    hspace=gap_h,
)

ax = [
    fig.add_subplot(gs[0, 0]),  # ax[0] — top-left
    fig.add_subplot(gs[0, 1]),  # ax[1] — top-right
    fig.add_subplot(gs[1, 0]),  # ax[2] — bottom-left
    fig.add_subplot(gs[1, 1]),  # ax[3] — bottom-right
]


alpha = 0.2
markersize = 6.0
ax[0].scatter(
    weight,
    fb_weight_sal,
    alpha=alpha,
    edgecolors="none",
    s=markersize,
    label="SAL",
    c="C0",
)
set_square_bound(ax[0], weight, fb_weight_sal)

ax[1].scatter(
    weight,
    fb_weight_rdd,
    alpha=alpha,
    edgecolors="none",
    s=markersize,
    label="RDD",
    c="C1",
)
set_square_bound(ax[1], weight, fb_weight_rdd)

ax[2].scatter(
    weight,
    fb_weight_stdwi,
    alpha=alpha,
    edgecolors="none",
    s=markersize,
    label="STDWI",
    c="C2",
)
set_square_bound(ax[2], weight, fb_weight_stdwi)

ax[3].plot(sal_res["time"], sal_res["angle"], linestyle="-", label="SAL")
ax[3].plot(rdd_res["time"], rdd_res["angle"], linestyle="--", label="RDD")
ax[3].plot(stdwi_res["time"], stdwi_res["angle"], linestyle="-.", label="STDWI")

ax[0].set_ylabel("B")
ax[2].set_xlabel("W")
ax[2].set_ylabel("B")

ax[3].set_xlabel(r"$t$ [s]")
ax[3].set_ylabel(r"$\measuredangle (\mathbf W^T, \mathbf B)\ [\mathrm{deg}]$")
ax[3].legend()

for a in ax[:3]:
    a.legend(loc="lower right")
    a.set_aspect("equal", adjustable="box")
    sync_ticks(a)
    a.grid()

ax[3].set_box_aspect(1)

In [ ]:
fig.savefig(FIG_DIR / "stdwi.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "stdwi.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "stdwi.svg", bbox_inches="tight")